#### Imports / configuration loading

In [1]:
# Reload imports if needed
# import source
# import importlib
# importlib.reload(source.results_analysis)
# importlib.reload(source.train_predictive_models)
# importlib.reload(source.predictive_models)
# importlib.reload(source.exploration)

In [2]:
from source import agents, exploration, predictive_models
from source.map_loader import Env
from source.train_predictive_models import process_input, runSGD, one_hot_encode
from source.results_analysis import metrics_predict_map,print_metrics,latent_space_PCA
from pathlib import Path
import source.results_analysis as results_analysis
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm
import json,re 
import pathlib
import os
import textwrap
import pandas as pd
import seaborn as sns

n_seeds_for_PCA = 5
n_seeds = 15
seeds = [77243, 798315, 965307, 159144, 101125, 843728, 77097, 814282, 151245, 698702, 206740, 38389, 363028, 379644, 429870]
settings = ["color_50","base_20","base_30","base_50"]
map_paths = {"base":"8x8_colors.txt","color":"8x8_colors.txt"}
observation_spaces = {"egocentric": exploration.exploration_egocentric,"allocentric":exploration.exploration_allocentric}

configs = []

config_dir = pathlib.Path("configs/exploration_comparison_tests/")
configs = [json.load(f.open()) for f in config_dir.glob("*.json")]

####  Data generation

In [3]:
config = configs[0]
colorbase = ["color", "base"]
data_keys = ['img_train', 'pos_train', 'dir_train', 'img_test', 'pos_test', 'dir_test']
loaded_data = {k1: {k2: {k3: None for k3 in seeds} for k2 in colorbase} for k1 in observation_spaces.keys()}
for observation_space in observation_spaces.keys():
    for c in colorbase:
        for seed in seeds:
            if observation_space == "egocentric":
                agent_type = ["random", [0.3333, 0.3333, 0.3334],seed]
            elif observation_space == "allocentric":
                agent_type = ["random", [0.25, 0.25, 0.25, 0.25], seed]

            with open(f"maps/{map_paths[c]}", 'r') as file:
                map = [[int(x) for x in line.strip().split(" ")] for line in file ]
            if (c=="color"):
                with open(f"maps/colors/{map_paths[c]}", 'r') as file:
                    color_map = [[int(x) for x in line.strip().split(" ")] for line in file ]
            else:
                color_map = None
            outfolder = f"exploration_data/exploration_comparison_tests/{observation_space}_{c}"
            outdir = f"{outfolder}/{seed}_exploration_data.npz"
            Path(outfolder).mkdir(parents=True, exist_ok=True)
            print(f"Setting:{c}, {observation_space}, {seed}")
            try:
                with np.load(outdir) as data:
                    loaded_data[observation_space][c][seed] = [data[k] for k in data_keys]
                    print ("Successfully loaded data!")
            except (FileNotFoundError, Exception):
                config['agent_type'][2] = seed
                env = Env(map,config['map_dims'],tuple(config['start_pos']),seed=seed,render_mode="rgb_array",agent_view_size=config['egocentric_view_size'],colors=color_map)
                env.reset(seed=env.env_seed)
                loaded_data[observation_space][c][seed] = observation_spaces[observation_space](
                env,
                agent_type=agent_type,
                n_steps_train=config['number_steps_train'],
                n_steps_test=config['number_steps_test'],
                n_restart_train=config['num_random_position_restarts_train'],
                n_restart_test=config['num_random_position_restarts_test'],
                with_colors=(c=="color"))
                exploration_data = loaded_data[observation_space][c][seed]
                np.savez_compressed(outdir, **dict(zip(data_keys, exploration_data)))
                print ("Exploration complete!")
# plt.imshow(env.render())
# plt.axis('off')
# plt.show()
#  env.gen_obs()['image'][:,:,0]

Setting:color, egocentric, 77243
Successfully loaded data!
Setting:color, egocentric, 798315
Successfully loaded data!
Setting:color, egocentric, 965307
Successfully loaded data!
Setting:color, egocentric, 159144
Successfully loaded data!
Setting:color, egocentric, 101125
Successfully loaded data!
Setting:color, egocentric, 843728
Successfully loaded data!
Setting:color, egocentric, 77097
Successfully loaded data!
Setting:color, egocentric, 814282
Successfully loaded data!
Setting:color, egocentric, 151245
Successfully loaded data!
Setting:color, egocentric, 698702
Successfully loaded data!
Setting:color, egocentric, 206740
Successfully loaded data!
Setting:color, egocentric, 38389
Successfully loaded data!
Setting:color, egocentric, 363028
Successfully loaded data!
Setting:color, egocentric, 379644
Successfully loaded data!
Setting:color, egocentric, 429870
Successfully loaded data!
Setting:base, egocentric, 77243
Successfully loaded data!
Setting:base, egocentric, 798315
Successfully

#### Predictive model training

In [4]:
config_names = [i["config_name"] for i in configs]
loaded_models = {k1: {k2: {k3: None for k3 in seeds} for k2 in settings} for k1 in config_names}
processed_inputs = {k1: {k2: {k3: None for k3 in seeds} for k2 in settings} for k1 in config_names}

for setting in tqdm(settings):
    for seed in tqdm(seeds):
        for config in tqdm(configs):
            MODEL_DIR = f"trained_models/exploration_comparison_tests/{config['predictive_model_path']}/{setting}"
            METRICS_DIR = f"outputs/exploration_comparison_tests/model_evaluation/{config['predictive_model_path']}/{setting}"
            # if config["model_type"]!="inputs":
            MODEL_PATH = f"{MODEL_DIR}/{seed}_model.pt"
            METRICS_PATH = f"{METRICS_DIR}/{seed}_model.pt"
            
            Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
            Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)
            image_list_train, pos_list_train, dir_list_train, image_list_test, pos_list_test, dir_list_test = loaded_data[config["observation_space"]]["color" if setting[:5] == "color" else "base"][seed]

            X_train, X_test = image_list_train, image_list_test 

            target_train = X_train[1:]
            input_train = X_train[:-1]
            target_test = X_test[1:]
            input_test = X_test[:-1]

            target_train = torch.tensor(target_train, dtype=torch.int64)
            input_train = torch.tensor(input_train, dtype=torch.int64)
            target_test = torch.tensor(target_test, dtype=torch.int64)
            input_test = torch.tensor(input_test, dtype=torch.int64)
            
            scaling_factor=1

            if config["observation_space"]=="allocentric":
                img_width, img_height = config["map_dims"][0], config["map_dims"][1]
            else:
                img_width, img_height = config["egocentric_view_size"], config["egocentric_view_size"]

            input_train_processed = torch.stack([process_input(x, 'in',img_width,img_height,scaling_factor) for x in input_train])
            input_test_processed = torch.stack([process_input(x, 'in',img_width,img_height,scaling_factor) for x in input_test])
            target_train_processed = torch.stack([process_input(x, 'out',img_width,img_height,scaling_factor) for x in target_train])
            target_test_processed = torch.stack([process_input(x, 'out',img_width,img_height,scaling_factor) for x in target_test])

            processed_inputs[config["config_name"]][setting][seed] = [input_train_processed, input_test_processed, target_train_processed, target_test_processed]

            # print(target_train_processed)
            # print(input_train_processed)

            match = re.search(r'(\d+)$', setting)
            latent_size = int(match.group(1)) 
            hidden_fac=15
            input_size=input_train_processed.size(1)
            output_size = target_train_processed.size(1)
            torch.manual_seed(config["model_seed"])
            torch.use_deterministic_algorithms(True)
            train_flag = True
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            if config['model_type'] == "autoencoder_no_rnn":
                model = predictive_models.AutoencoderWithoutRNN(input_size, output_size, hidden_fac, latent_size,device)
            elif config["model_type"]=="autoencoder_rnn":
                model = predictive_models.AutoencoderWithRNN(input_size, output_size, hidden_fac, latent_size,device)
            elif config["model_type"]=="autoencoder_rnn_separate_action":
                model = predictive_models.AutoencoderRNNSeparateAction(img_width*img_height,input_size-img_width*img_height, output_size, hidden_fac, latent_size,device)
            else:
                model = predictive_models.AutoencoderInputs(img_width*img_height,device)
                train_flag = False
            n_epochs, batch_size = config['number_epochs'], config['batch_size']
            if (config["load_trained_model_if_existent"] and os.path.exists(MODEL_PATH)) and config["model_type"]!="inputs":
                try:
                    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
                    print("Loaded model!")
                    with open(f"outputs/exploring_latent_space/model_evaluation/{config['config_name']}.json", "r") as f:
                        metrics = json.load(f)
                    train_flag = False
                except (RuntimeError, KeyError, EOFError) as e:
                    print("Failed to load model - will proceed to training!")
            if train_flag:
                tracked_losses_train, tracked_losses_test = runSGD(model, input_train_processed, target_train_processed, input_test_processed, target_test_processed, device=device, n_epochs=n_epochs, batch_size=batch_size,seed=config["model_seed"],notrain=(not config["train"]))
                if (config['save_trained_model']):
                    torch.save(model.state_dict(), MODEL_PATH)
                
                metrics_predict_map(None,model,config,device,scaling_factor,input_test_processed,target_test,tracked_losses_train,tracked_losses_test,outdir=METRICS_PATH,allo_width=img_width,allo_height=img_height,ego_len=img_width)
                
            loaded_models[config["config_name"]][setting][seed] = model 

  0%|          | 0/4 [00:00<?, ?it/s]
/home/likescience/SURFIN/predictive_model_latent_space/source/train_predictive_models.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  world_map = torch.tensor(input_array[:-1]).float()
/home/likescience/SURFIN/predictive_model_latent_space/source/train_predictive_models.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(input_array[:-1])/scaling_factor





Loaded model!


Loaded model!



100%|██████████| 5/5 [00:01<00:00,  3.65it/s]


Loaded model!


...

Loaded model!


Loaded model!


Loaded model!



100%|██████████| 4/4 [01:11<00:00, 17.93s/it]

Loaded model!


#### PCA on latent space 

In [ ]:
top_components = {k1: {k2: {k3: [] for k3 in seeds} for k2 in settings} for k1 in config_names}

for setting in tqdm(settings):
    for seed in tqdm(seeds[:n_seeds_for_PCA]):
        for config in tqdm(configs):
            # print(setting,seed,config)
            OUT_DIR = f"outputs/exploration_comparison_tests/latent_space_PCA/{config['predictive_model_path']}/{setting}/{seed}"
            input_train_processed = processed_inputs[config["config_name"]][setting][seed][0]
            image_list_train, pos_list_train, dir_list_train, image_list_test, pos_list_test, dir_list_test = loaded_data[config["observation_space"]]["color" if setting[:5] == "color" else "base"][seed]
            model = loaded_models[config["config_name"]][setting][seed]
            # latent_space_before, latent_space_after = model.get_latent(input_train_processed)
            all_latent_spaces = [(model.get_latent(i.view(1,-1))) for i in input_train_processed]
            latent_space_before = torch.stack([i[0] for i in all_latent_spaces]).squeeze(1)
            latent_space_after_list = [i[1] for i in all_latent_spaces]
            if all(val is not None for val in latent_space_after_list):
                latent_space_after = torch.stack(latent_space_after_list).squeeze(1)
            else:
                latent_space_after = None

            if (config["model_type"] == "autoencoder_rnn"):
                latent_spaces = [[latent_space_before, "before_RNN"],[latent_space_after, "after_RNN"]]
            else:
                latent_spaces = [[latent_space_before, ""]]
            print(config["config_name"],setting,seed)
            top_components[config["config_name"]][setting][seed] = latent_space_PCA(latent_spaces,config,pos_list_train[:-1],dir_list_train[:-1],img_width,img_height,outdir=OUT_DIR)



  0%|          | 0/4 [00:00<?, ?it/s]


allocentric_8x8_empty_inputs color_50 77243


egocentric_8x8_empty_7x7view_inputs color_50 77243


egocentric_8x8_empty_7x7view_rnn_separate_action color_50 77243


allocentric_8x8_empty_rnn color_50 77243


...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

#### Linear regression on the first PCA components

In [9]:
for setting in tqdm(settings):
    for seed in tqdm(seeds[:n_seeds_for_PCA]):
        for config in tqdm(configs):
            OUT_DIR = f"outputs/exploration_comparison_tests/PCA_linear_regression/{config['config_name']}/{setting}/{seed}"
            Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
            input_train_processed = processed_inputs[config["config_name"]][setting][seed][0]
            image_list_train, pos_list_train, dir_list_train, image_list_test, pos_list_test, dir_list_test = loaded_data[config["observation_space"]]["color" if setting[:5] == "color" else "base"][seed]
            model = loaded_models[config["config_name"]][setting][seed]
            latent_space_before, latent_space_after = model.get_latent(input_train_processed)
            if (config["model_type"] == "autoencoder_rnn"):
                top_principal_components = [[top_components[config["config_name"]][setting][seed][0], latent_space_before, "before_RNN"],[top_components[config["config_name"]][setting][seed][1], latent_space_after, "after_RNN"]]
            else:
                top_principal_components = [[top_components[config["config_name"]][setting][seed][0], latent_space_before, ""]]
            for top_comps, latent_space, label in top_principal_components:
                X = latent_space @ top_comps #X: N x 3
                X_wbias = torch.cat([X, torch.ones(X.shape[0], 1)], dim=1) #X_wbias: N x 4
                results_reg = {}
                results_reg["label"] = f"{label}"
                for metric in ["x_position","y_position","L2_dist_center"]: #y: Nx1
                    if metric == "x_position":
                        y = torch.tensor(np.array(pos_list_train)[:-1,0]).float()
                    elif metric == "y_position":
                        y = torch.tensor(np.array(pos_list_train)[:-1,1]).float()
                    elif metric == "L2_dist_center":
                        y = torch.tensor(np.sqrt((np.array(pos_list_train)[:-1,0]-(img_width-1)/2)**2+ (np.array(pos_list_train)[:-1,1]-(img_height-1)/2)**2)).float()
                    y = y.view(-1,1)
                    W = torch.linalg.pinv(X_wbias) @ y # W: 4 x 1
                    y_pred = X_wbias @ W # y_pred: N x 1
                    r_sq = 1-torch.sum((y-y_pred)**2)/torch.sum((y-y.mean())**2)
                    results_reg[f"R^2 of {metric}"] = r_sq.item()
                with open(f"{OUT_DIR}/{label}_rsq.json", "w") as f:
                    json.dump(results_reg, f, indent=4)
                # print(r_sq)
                    

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 14.25it/s]


100%|██████████| 5/5 [00:00<00:00, 17.57it/s]


100%|██████████| 5/5 [00:00<00:00, 15.24it/s]


100%|██████████| 5/5 [00:00<00:00, 14.35it/s]


 25%|██▌       | 1/4 [00:01<00:05,  1.81s/it]

100%|██████████| 5/5 [00:00<00:00, 15.08it/s]


100%|██████████| 5/5 [00:00<00:00, 16.69it/s]


100%|██████████| 5/5 [00:00<00:00, 17.20it/s]


100%|██████████| 5/5 [00:00<00:00, 15.77it/s]


 50%|█████     | 2/4 [00:03<00:03,  1.67s/it]

100%|██████████| 5/5 [00:00<00:00, 13.08it/s]


100%|██████████| 5/5 [00:00<00:00, 15.41it/s]


100%|██████████| 5/5 [00:00<00:00, 16.42it/s]


100%|██████████| 5/5 [00:00<00:00, 18.96it/s]


 75%|███████▌  | 3/4 [00:04<00:01,  1.64s/it]

100%|██████████| 5/5 [00:00<00:00, 18.65it/s]


100%|██████████| 5/5 [00:00<00:00, 18.60it/s]


100%|██████████| 5/5 [00:00<00:00, 18.39it/s]


100%|██████████| 5/5 [00:00<00:00, 15.83it/s]


100%|██████████| 4/4 [00:06<00:00,  1.60s/it]


##### Violin plots

In [10]:
data_directory = 'outputs/exploration_comparison_tests/PCA_linear_regression'
target_vars = ["x_position","y_position","L2_dist_center"]
target_vars_labels = {"x_position": "X coordinate", "y_position": "Y coordinate", "L2_dist_center":"L2 distance to environment center"}

for target_var in target_vars: 
    OUT_DIR_RSQP = f"outputs/exploration_comparison_tests/PCA_linear_regression/{target_var}"
    Path(OUT_DIR_RSQP).mkdir(parents=True, exist_ok=True)
    labels_config = {k1: {k2: {k3: None for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    labels_setting = {k1: {k2: {k3: None for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    all_values = {k1: {k2: {k3: [] for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    metric_label = ""
    configs_before_after_rnn =  ["egocentric_8x8_empty_7x7view_rnn", "allocentric_8x8_empty_rnn"]

    for config_name in sorted(os.listdir(data_directory),reverse=True):
        # print(config_name, configs_before_after_rnn[0], config_name in configs_before_after_rnn)
        if config_name not in target_vars:
            for setting in sorted(os.listdir(f"{data_directory}/{config_name}"),reverse=True):
                for seed in sorted(os.listdir(f"{data_directory}/{config_name}/{setting}"),reverse=True):
                    for filename in sorted(os.listdir(f"{data_directory}/{config_name}/{setting}/{seed}"),reverse=True):
                        if filename.endswith('.json'):
                            path = os.path.join(Path(f"{data_directory}/{config_name}/{setting}/{seed}"), filename)
                            with open(path, 'r') as f:
                                data = json.load(f)                                
                                metric_label = "R^2"
                                relative_rnn = "before"
                                if config_name in configs_before_after_rnn and filename[:5] == "after":
                                    relative_rnn = "after"
                                all_values[config_name][setting][relative_rnn] += [data[f"R^2 of {target_var}"]]
                                labels_config[config_name][setting][relative_rnn] = f"{config_name.replace('_', ' ')} {data['label']}"
                                labels_setting[config_name][setting][relative_rnn] = setting
    
    for config_name in config_names:
        if config_name in configs_before_after_rnn:
            relative_rnn_latent_space = [["before","_before_RNN"],["after","_after_RNN"]]
        else:
            relative_rnn_latent_space = [["before",""]]
        for relative_rnn,file_label in relative_rnn_latent_space:
            labels = []
            values = []
            for setting in settings:
                labels.append(labels_setting[config_name][setting][relative_rnn])
                values.append(all_values[config_name][setting][relative_rnn])
            # print(target_var, labels)
            # print(relative_rnn,file_label)
            wrapped_labels = [textwrap.fill(label, width=30) for label in labels]
            df = pd.DataFrame([{'L': l, 'V': v} for l, vs in zip(wrapped_labels, values) for v in vs])
            plt.figure(figsize=(8, 6))
            sns.violinplot(data=df, x='V', y='L',hue= 'L', palette='viridis', inner='box', linewidth=1.5)
            plt.ylabel('Model type'); plt.xlabel(f"{target_vars_labels[target_var]} {metric_label}")
            plt.savefig(f"{OUT_DIR_RSQP}/{config_name}{file_label}.png", bbox_inches='tight'); plt.clf()

    for setting in settings:
        labels = []
        values = []
        for config_name in config_names:
            if config_name in configs_before_after_rnn:
                relative_rnn_latent_space = ["before","after"]
            else:
                relative_rnn_latent_space = ["before"]
            for relative_rnn in relative_rnn_latent_space:
                labels.append(labels_config[config_name][setting][relative_rnn])
                values.append(all_values[config_name][setting][relative_rnn])
        wrapped_labels = [textwrap.fill(label, width=30) for label in labels]
        df = pd.DataFrame([{'L': l, 'V': v} for l, vs in zip(wrapped_labels, values) for v in vs])
        plt.figure(figsize=(8, 6))
        sns.violinplot(data=df, x='V', y='L',hue= 'L', palette='viridis', inner='box', linewidth=1.5)
        plt.ylabel('Model type'); plt.xlabel(f"{target_vars_labels[target_var]} {metric_label}")
        plt.savefig(f"{OUT_DIR_RSQP}/{setting}.png", bbox_inches='tight'); plt.clf()


/tmp/ipykernel_121653/520400504.py:65: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(8, 6))


<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

#### Training a single-layer decoder to decode latent space representaitons

In [ ]:
for setting in tqdm(settings):
    for seed in tqdm(seeds):
        for config in tqdm(configs):
            model = loaded_models[config["config_name"]][setting][seed]
            input_train_processed, input_test_processed = processed_inputs[config["config_name"]][setting][seed][:2]
            image_list_train, pos_list_train, dir_list_train, image_list_test, pos_list_test, dir_list_test = loaded_data[config["observation_space"]]["color" if setting[:5] == "color" else "base"][seed]
            if config["observation_space"]=="allocentric":
                img_width, img_height = config["map_dims"][0], config["map_dims"][1]
            else:
                img_width, img_height = config["egocentric_view_size"], config["egocentric_view_size"]
            OUT_DIR_REPR = f"outputs/exploration_comparison_tests/decode_latent_representations_1layer/{config['config_name']}/{setting}/{seed}"
            match = re.search(r'(\d+)$', setting)
            latent_size = int(match.group(1)) 
            Path(OUT_DIR_REPR).mkdir(parents=True, exist_ok=True)
            
            latent_space_before_train, latent_space_after_train = model.get_latent(input_train_processed)
            latent_space_before_test, latent_space_after_test = model.get_latent(input_test_processed)

            if (config["model_type"] == "autoencoder_rnn"):
                latent_size_repr = latent_size
                latent_representations_train = [[latent_space_before_train.detach(), "_before_RNN"],[latent_space_after_train.detach(), "_after_RNN"]]
                latent_representations_test = [[latent_space_before_test.detach(), "_before_RNN"],[latent_space_after_test.detach(), "_after_RNN"]]
            elif (config["model_type"] == "autoencoder_no_rnn"):
                latent_size_repr = latent_size
                latent_representations_train = [[latent_space_before_train.detach(), ""]]
                latent_representations_test = [[latent_space_before_test.detach(), ""]]
            elif (config["model_type"] == "autoencoder_rnn_separate_action"):
                latent_size_repr = latent_size-(input_size-img_height*img_width)
                latent_representations_train = [[latent_space_before_train.detach(), ""]]
                latent_representations_test = [[latent_space_before_test.detach(), ""]]
            elif (config["model_type"] == "inputs"):
                latent_size_repr = img_height*img_width
                latent_representations_train = [[latent_space_before_train.detach(), ""]]
                latent_representations_test = [[latent_space_before_test.detach(), ""]]

            dads = []
            for i in range(len(latent_representations_train)):
                latent_representation_train, label = latent_representations_train[i]
                latent_representation_test, label = latent_representations_test[i]
                if not Path(f"{OUT_DIR_REPR}/{label}.json").exists():
                    dict_repr = {}
                    for variable in config["variables_for_training_decoder"]:
                        if variable == "position":
                                target_train_repr, target_test_repr = torch.tensor(pos_list_train[:-1]).float(), torch.tensor(pos_list_test[:-1]).float()
                                decoder_model = nn.Linear(latent_size_repr,2)
                                loss_type="mse"
                                metric, metric_label, output_process = results_analysis.accuracy, "accuracy",results_analysis.round
                        elif variable == "L2_dist_center":
                            target_train_repr = torch.tensor(np.sqrt((np.array(pos_list_train)[:-1,0]-(img_width-1)/2)**2+ (np.array(pos_list_train)[:-1,1]-(img_height-1)/2)**2)).float().unsqueeze(-1)
                            target_test_repr = torch.tensor(np.sqrt((np.array(pos_list_test)[:-1,0]-(img_width-1)/2)**2+ (np.array(pos_list_test)[:-1,1]-(img_height-1)/2)**2)).float().unsqueeze(-1)
                            decoder_model = nn.Linear(latent_size_repr,1)
                            loss_type="mse"
                            metric, metric_label, output_process = results_analysis.mse, "mse",results_analysis.identity
                        elif variable == "head_direction":
                            target_train_repr, target_test_repr = torch.tensor(dir_list_train[:-1]).long().view(-1), torch.tensor(dir_list_test[:-1]).long().view(-1)
                            decoder_model = nn.Linear(latent_size_repr,4)
                            loss_type="cel"
                            metric, metric_label, output_process = results_analysis.accuracy, "accuracy", results_analysis.argmax
                        tracked_losses_train_repr, tracked_losses_test_repr = runSGD(decoder_model,latent_representation_train,target_train_repr,latent_representation_test,target_test_repr,device,criterion=loss_type,n_epochs=100,batch_size=256,shuffle=False,hide_plot=True,lr=.01)
                        output_test_repr = decoder_model(latent_representation_test)
                        output_test_repr = output_process(output_test_repr)
                        results_repr = {}
                        results_repr["name_config"] = f"{config['config_name']}{label}"
                        results_repr["name_setting"] = f"{setting}{label}"
                        results_repr["metric"] = metric(output_test_repr, target_test_repr)
                        results_repr["metric_name"] = metric_label
                        results_repr["tracked_losses_test"] = tracked_losses_test_repr
                        dict_repr[variable] = results_repr
                        print(results_repr)
                        print(decoder_model.weight.grad.abs().mean())
                        dads += [output_test_repr,target_test_repr]
                    
                    with open(f"{OUT_DIR_REPR}/{label}.json", "w") as f:
                        json.dump(dict_repr, f, indent=4)


  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 15.71it/s]


100%|██████████| 5/5 [00:00<00:00, 13.74it/s]


100%|██████████| 5/5 [00:00<00:00, 19.80it/s]


100%|██████████| 5/5 [00:00<00:00, 20.34it/s]


100%|██████████| 5/5 [00:00<00:00, 23.06it/s]


100%|██████████| 5/5 [00:00<00:00, 24.22it/s]


100%|██████████| 5/5 [00:00<00:00, 22.99it/s]



Epoch 	 Loss train 	 Loss test
1/100	 3.3841		 2.9977
2/100	 3.2177		 3.1665
3/100	 3.1019		 3.1934
4/100	 3.0426		 3.2239
5/100	 3.0050		 3.2551
6/100	 2.9753		 3.2660
7/100	 2.9507		 3.2658
8/100	 2.9294		 3.2581
9/100	 2.9104		 3.2461
10/100	 2.8935		 3.2316
11/100	 2.8782		 3.2161
12/100	 2.8645		 3.2003
13/100	 2.8520		 3.1846
14/100	 2.8407		 3.1693
15/100	 2.8304		 3.1545
16/100	 2.8210		 3.1402
17/100	 2.8124		 3.1265
18/100	 2.8046		 3.1134
19/100	 2.7974		 3.1009
20/100	 2.7908		 3.0889
21/100	 2.7848		 3.0774
22/100	 2.7792		 3.0664
23/100	 2.7741		 3.0559
24/100	 2.7693		 3.0459
25/100	 2.7649		 3.0362
26/100	 2.7608		 3.0270
27/100	 2.7571		 3.0182
28/100	 2.7535		 3.0097
29/100	 2.7503		 3.0016
30/100	 2.7472		 2.9939
31/100	 2.7443		 2.9864
32/100	 2.7416		 2.9793
33/100	 2.7391		 2.9724
34/100	 2.7368		 2.9658
35/100	 2.7345		 2.9594
36/100	 2.7324		 2.9533
37/100	 2.7305		 2.9474
38/100	 2.7286		 2.9418
39/100	 2.7268		 2.9363
40/100	 2.7251		 2.9311
41/100	 2.7235		 2

95/100	 1.2458		 1.0886
96/100	 1.2457		 1.0887
97/100	 1.2457		 1.0887
98/100	 1.2456		 1.0887
99/100	 1.2455		 1.0887
100/100	 1.2455		 1.0888
{'name_config': 'egocentric_8x8_empty_7x7view_inputs', 'name_setting': 'color_50', 'metric': 0.6238119006156921, 'metric_name': 'accuracy', 'tracked_losses_test': [1.2409099340438843, 1.1688240766525269, 1.1443126201629639, 1.1289105415344238, 1.1187052726745605, 1.1120193004608154, 1.1071196794509888, 1.1034196615219116, 1.1005579233169556, 1.0982978343963623, 1.096482515335083, 1.0950052738189697, 1.0937893390655518, 1.0927796363830566, 1.0919348001480103, 1.0912234783172607, 1.090622067451477, 1.090111255645752, 1.0896759033203125, 1.089303970336914, 1.0889856815338135, 1.0887126922607422, 1.0884788036346436, 1.0882781744003296, 1.0881057977676392, 1.0879584550857544, 1.0878324508666992, 1.0877251625061035, 1.087633728981018, 1.0875569581985474, 1.0874922275543213, 1.0874383449554443, 1.087394118309021, 1.0873583555221558, 1.087330222129821

100/100	 1.2236		 1.1287
{'name_config': 'egocentric_8x8_empty_7x7view_rnn_separate_action', 'name_setting': 'color_50', 'metric': 0.6148074269294739, 'metric_name': 'accuracy', 'tracked_losses_test': [1.3036839962005615, 1.201229453086853, 1.1867239475250244, 1.1719213724136353, 1.1614688634872437, 1.1541768312454224, 1.1487091779708862, 1.1444153785705566, 1.1409436464309692, 1.1380738019943237, 1.1356611251831055, 1.1336058378219604, 1.1318379640579224, 1.1303050518035889, 1.1289677619934082, 1.127795696258545, 1.126764178276062, 1.1258538961410522, 1.125049114227295, 1.1243360042572021, 1.123704195022583, 1.1231443881988525, 1.1226481199264526, 1.122208833694458, 1.1218209266662598, 1.1214790344238281, 1.121179223060608, 1.1209170818328857, 1.1206893920898438, 1.120492935180664, 1.1203253269195557, 1.1201838254928589, 1.12006676197052, 1.1199716329574585, 1.1198970079421997, 1.11984121799469, 1.1198028326034546, 1.1197803020477295, 1.1197729110717773, 1.1197792291641235, 1.11979806

88/100	 0.0791		 0.1433
89/100	 0.0784		 0.1430
90/100	 0.0777		 0.1427
91/100	 0.0770		 0.1424
92/100	 0.0764		 0.1421
93/100	 0.0757		 0.1418
94/100	 0.0751		 0.1415
95/100	 0.0745		 0.1413
96/100	 0.0739		 0.1410
97/100	 0.0733		 0.1408
98/100	 0.0727		 0.1405
99/100	 0.0721		 0.1403
100/100	 0.0715		 0.1401
{'name_config': 'allocentric_8x8_empty_rnn_after_RNN', 'name_setting': 'color_50_after_RNN', 'metric': 0.9804902672767639, 'metric_name': 'accuracy', 'tracked_losses_test': [1.0471841096878052, 0.8949109315872192, 0.7643833160400391, 0.6684705018997192, 0.593858003616333, 0.534763753414154, 0.48725998401641846, 0.4485047161579132, 0.4164258539676666, 0.3894897997379303, 0.3665729761123657, 0.3468572199344635, 0.3297431170940399, 0.31478002667427063, 0.30161619186401367, 0.2899678945541382, 0.27960205078125, 0.27032580971717834, 0.261980265378952, 0.2544343173503876, 0.24758043885231018, 0.24132941663265228, 0.23560744524002075, 0.2303524762392044, 0.22551171481609344, 0.22104008

...

99/100	 1.2767		 1.2269
100/100	 1.2766		 1.2271
{'name_config': 'egocentric_8x8_empty_7x7view_inputs', 'name_setting': 'base_50', 'metric': 0.5192596316337585, 'metric_name': 'accuracy', 'tracked_losses_test': [1.3639549016952515, 1.2588080167770386, 1.2241692543029785, 1.2070095539093018, 1.1969244480133057, 1.1910250186920166, 1.1875869035720825, 1.1857857704162598, 1.1850882768630981, 1.1851472854614258, 1.1857248544692993, 1.1866527795791626, 1.1878106594085693, 1.1891103982925415, 1.1904886960983276, 1.1918995380401611, 1.193310022354126, 1.1946972608566284, 1.1960453987121582, 1.1973437070846558, 1.198586106300354, 1.1997690200805664, 1.2008907794952393, 1.201952338218689, 1.2029544115066528, 1.2038992643356323, 1.2047895193099976, 1.2056280374526978, 1.2064176797866821, 1.2071616649627686, 1.2078630924224854, 1.2085245847702026, 1.2091491222381592, 1.2097398042678833, 1.2102984189987183, 1.2108278274536133, 1.211330533027649, 1.211808204650879, 1.2122629880905151, 1.21269679069

99/100	 1.2246		 1.1893
100/100	 1.2245		 1.1893
{'name_config': 'egocentric_8x8_empty_7x7view_rnn_separate_action', 'name_setting': 'base_50', 'metric': 0.5082541108131409, 'metric_name': 'accuracy', 'tracked_losses_test': [1.36744225025177, 1.302834391593933, 1.2790127992630005, 1.2628141641616821, 1.253034234046936, 1.245632529258728, 1.2396172285079956, 1.2346497774124146, 1.2304866313934326, 1.2269556522369385, 1.2239316701889038, 1.2213183641433716, 1.219041347503662, 1.2170398235321045, 1.215266227722168, 1.2136820554733276, 1.2122560739517212, 1.2109631299972534, 1.2097831964492798, 1.2086999416351318, 1.2076992988586426, 1.206770896911621, 1.2059049606323242, 1.205094575881958, 1.2043331861495972, 1.2036159038543701, 1.202938437461853, 1.2022966146469116, 1.2016874551773071, 1.201108694076538, 1.2005574703216553, 1.2000319957733154, 1.199530839920044, 1.1990516185760498, 1.1985939741134644, 1.1981559991836548, 1.1977369785308838, 1.1973354816436768, 1.1969507932662964, 1.19658

99/100	 0.2445		 0.3068
100/100	 0.2433		 0.3059
{'name_config': 'allocentric_8x8_empty_rnn_after_RNN', 'name_setting': 'base_50_after_RNN', 'metric': 0.9169584512710571, 'metric_name': 'accuracy', 'tracked_losses_test': [1.1975854635238647, 1.0885273218154907, 1.0079090595245361, 0.9406141638755798, 0.8828057646751404, 0.8340172171592712, 0.7925541400909424, 0.7568525671958923, 0.7258515954017639, 0.6986920833587646, 0.6746930480003357, 0.6533129811286926, 0.6341202259063721, 0.6167672872543335, 0.6009745001792908, 0.5865175127983093, 0.5732154250144958, 0.5609211921691895, 0.5495132803916931, 0.538890540599823, 0.5289676189422607, 0.519672155380249, 0.5109415054321289, 0.5027220845222473, 0.49496689438819885, 0.48763513565063477, 0.480690598487854, 0.4741012752056122, 0.46783894300460815, 0.4618781507015228, 0.4561963677406311, 0.45077309012413025, 0.4455900192260742, 0.4406304359436035, 0.43587934970855713, 0.4313230812549591, 0.4269489645957947, 0.42274585366249084, 0.4187030792236


100%|██████████| 4/4 [21:18<00:00, 319.71s/it]

98/100	 1.1930		 1.2282
99/100	 1.1928		 1.2282
100/100	 1.1926		 1.2282
{'name_config': 'egocentric_8x8_empty_7x7view_rnn_after_RNN', 'name_setting': 'base_50_after_RNN', 'metric': 0.42771387100219727, 'metric_name': 'accuracy', 'tracked_losses_test': [1.3497998714447021, 1.317298412322998, 1.2972692251205444, 1.2833149433135986, 1.2729166746139526, 1.264715313911438, 1.2581232786178589, 1.2527817487716675, 1.2484139204025269, 1.2448155879974365, 1.2418335676193237, 1.239349603652954, 1.2372715473175049, 1.23552668094635, 1.2340573072433472, 1.2328165769577026, 1.2317672967910767, 1.2308783531188965, 1.230125069618225, 1.229486107826233, 1.2289447784423828, 1.2284858226776123, 1.228097915649414, 1.2277700901031494, 1.2274943590164185, 1.2272629737854004, 1.227069616317749, 1.2269089221954346, 1.226776123046875, 1.2266677618026733, 1.2265801429748535, 1.2265102863311768, 1.226455807685852, 1.2264143228530884, 1.2263844013214111, 1.226364016532898, 1.2263519763946533, 1.2263469696044922

#### Decode latent representation violin plots

In [66]:
directory = 'outputs/exploration_comparison_tests/decode_latent_representations_1layer'
dirs_to_exclude = {"head_direction": ["allocentric_8x8_empty_rnn", "allocentric_8x8_empty_untrained_rnn","allocentric_8x8_empty_untrained","allocentric_8x8_empty.json"] }
target_vars = ["position","L2_dist_center", "head_direction"]
target_vars_labels = {"position": "Position", "L2_dist_center":"L2 distance to environment center","head_direction": "Head Direction" }

for target_var in target_vars: 
    OUT_DIR_EXP = f"outputs/exploration_comparison_tests/decode_latent_representations_1layer/{target_var}"
    Path(OUT_DIR_EXP).mkdir(parents=True, exist_ok=True)
    labels_config = {k1: {k2: {k3: None for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    labels_setting = {k1: {k2: {k3: None for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    all_values = {k1: {k2: {k3: [] for k3 in ["before","after"]} for k2 in settings} for k1 in config_names}
    metric_label = ""
    configs_before_after_rnn =  ["egocentric_8x8_empty_7x7view_rnn", "allocentric_8x8_empty_rnn"]

    for config_name in sorted(os.listdir(directory),reverse=True):
        # print(config_name, configs_before_after_rnn[0], config_name in configs_before_after_rnn)
        if config_name not in target_vars:
            for setting in sorted(os.listdir(f"{directory}/{config_name}"),reverse=True):
                for seed in sorted(os.listdir(f"{directory}/{config_name}/{setting}"),reverse=True):
                    for filename in sorted(os.listdir(f"{directory}/{config_name}/{setting}/{seed}"),reverse=True):
                        if filename.endswith('.json'):
                            if (target_var in dirs_to_exclude.keys() and filename.startswith(tuple(dirs_to_exclude[target_var]))):
                                continue
                            path = os.path.join(Path(f"{directory}/{config_name}/{setting}/{seed}"), filename)
                            with open(path, 'r') as f:
                                data = json.load(f)
                                if target_var in data:
                                    metric_label = data[target_var]["metric_name"]
                                    relative_rnn = "before"
                                    if config_name in configs_before_after_rnn and filename[-14:-9] == "after":
                                        relative_rnn = "after"
                                    all_values[config_name][setting][relative_rnn] += [data[target_var]["metric"]]
                                    labels_config[config_name][setting][relative_rnn] = data[target_var]["name_config"].replace("_", " ")
                                    labels_setting[config_name][setting][relative_rnn] = data[target_var]["name_setting"].replace("_", " ")
    
    for config_name in config_names:
        if config_name in configs_before_after_rnn:
            relative_rnn_latent_space = [["before","_before_RNN"],["after","_after_RNN"]]
        else:
            relative_rnn_latent_space = [["before",""]]
        for relative_rnn,file_label in relative_rnn_latent_space:
            labels = []
            values = []
            for setting in settings:
                labels.append(labels_setting[config_name][setting][relative_rnn])
                values.append(all_values[config_name][setting][relative_rnn])
            # print(target_var, labels)
            # print(relative_rnn,file_label)
            wrapped_labels = [textwrap.fill(label, width=30) for label in labels]
            df = pd.DataFrame([{'L': l, 'V': v} for l, vs in zip(wrapped_labels, values) for v in vs])
            plt.figure(figsize=(8, 6))
            sns.violinplot(data=df, x='V', y='L',hue= 'L', palette='viridis', inner='box', linewidth=1.5)
            plt.ylabel('Model type'); plt.xlabel(f"{target_vars_labels[target_var]} {metric_label}")
            plt.savefig(f"{OUT_DIR_EXP}/{config_name}{file_label}.png", bbox_inches='tight'); plt.clf()

    for setting in settings:
        labels = []
        values = []
        for config_name in config_names:
            if config_name in configs_before_after_rnn:
                relative_rnn_latent_space = ["before","after"]
            else:
                relative_rnn_latent_space = ["before"]
            for relative_rnn in relative_rnn_latent_space:
                labels.append(labels_config[config_name][setting][relative_rnn])
                values.append(all_values[config_name][setting][relative_rnn])
        wrapped_labels = [textwrap.fill(label, width=30) for label in labels]
        df = pd.DataFrame([{'L': l, 'V': v} for l, vs in zip(wrapped_labels, values) for v in vs])
        plt.figure(figsize=(8, 6))
        sns.violinplot(data=df, x='V', y='L',hue= 'L', palette='viridis', inner='box', linewidth=1.5)
        plt.ylabel('Model type'); plt.xlabel(f"{target_vars_labels[target_var]} {metric_label}")
        plt.savefig(f"{OUT_DIR_EXP}/{setting}.png", bbox_inches='tight'); plt.clf()


/tmp/ipykernel_787/852762977.py:69: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(8, 6))


<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>